imports

In [6]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from typing import TypedDict,Literal
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field



load_dotenv()


True

In [3]:


generator_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    api_key=os.getenv("gemini-api-key"),
    temperature=0.7
)

evaluator_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    api_key=os.getenv("gemini-api-key"),
    temperature=0.7
)


optimizer_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    api_key=os.getenv("gemini-api-key"),
    temperature=0.7
)

In [10]:
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation:Literal["approved","needs_approvement"]
    feedback:str
    iteration:int
    max_iterations:int


class TweetEvaluation(BaseModel):
    evaluation:Literal["approved","needs_approvement"]=Field(...,description="Whether the tweet is approved or needs improvement")
    feedback:str = Field(...,description="Feedback for improving the tweet if evaluation is 'needs_approvement'")
    

In [9]:
def generate_tweet(state: TweetState):
    messages=[
        SystemMessage(content="you are a funny and clever Twitter/X influncer"),
        HumanMessage(content=f"""write a short ,original and hilarious tweet on the topic : {state['topic']}

        rules:
        - the tweet should be less than 280 characters
        -don't use question answer format 
        -think in meme lgoic ,punchlines or releatable takes.
        -use simple day to day english
"""),
    ]

    response=generator_llm.invoke(messages).content
    return {"tweet":response}


def evaluate_tweet(state: TweetState):
    messages=[
        SystemMessage(content="you are a strict but fair Twitter/X content moderator"),
        HumanMessage(content=f"""evaluate the following tweet: {state['tweet']} on the topic {state['topic']} and determine if it's good enough to be posted without any changes. 

        rules:
        - if the tweet is hilarious, original and relevant to the topic, return "approved"
        - if the tweet is not funny, not original or not relevant to the topic, return "needs_approvement" and provide constructive feedback on how to improve it.
"""),
    ]

    structure_evavluator=evaluator_llm.with_structured_output(TweetEvaluation)

    response=structure_evavluator.invoke(messages)
    return {"evaluation":response.evaluation,"feedback":response.feedback}


def optimize_tweet(state: TweetState):
    messages=[
        SystemMessage(content="you are a talented comedy writer who is great at improving jokes and tweets"),
        HumanMessage(content=f"""improve the following tweet based on the feedback provided. 

        topic: {state['topic']}
        original tweet: {state['tweet']}
        feedback from moderator: {state['feedback']}

        rules for improvement:
        - make the tweet more hilarious, original and relevant to the topic
        - keep the tweet under 280 characters
        - use simple day to day english
"""),
    ]

    response=optimizer_llm.invoke(messages).content
    iteration=state['iteration']+1
    return {"tweet":response,"iteration":iteration}

def route_evevluaiton(state: TweetState):
    if state['evaluation']=="approved"   or state['iteration']>=state['max_iterations']:
        return "approved"
    
    else:
        return "needs_improvement"
    
       
    





In [20]:
graph=StateGraph(TweetState)

# add nodes
graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("optimize_tweet", optimize_tweet)

# add edges
graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet", route_evevluaiton,{'approved':END,"needs_improvement":"optimize_tweet"})
graph.add_edge("optimize_tweet", "evaluate_tweet")

workflow=graph.compile()
initial_state = {
    "topic": "virat kohli epic roast",
    "iteration": 1,
    "max_iterations": 3
}

final_state=workflow.invoke(initial_state)
final_state

{'topic': 'virat kohli epic roast',
 'tweet': 'Here are a few options for the improved tweet, aiming for more originality and humor:\n\n**Option 1 (Focus on the awards\' confusion):**\n\n> Virat Kohli\'s individual awards recently attended a seminar on \'Team Sports & Collective Glory.\' They left more confused than ever, asking, "So, you\'re saying *everyone* gets one? Not just the guy with the most hundreds?" 😂 #EpicRoast #KingKohli\n\n**Option 2 (More active search):**\n\n> Virat Kohli\'s individual awards have reportedly started a \'Missing Persons\' campaign for the elusive \'Team Trophy.\' They\'ve put up posters, but Kohli just keeps bringing home another MVP medal, completely missing the point. 🤦\u200d♂️ #EpicRoast #KingKohli\n\n**Option 3 (Slightly absurd, focusing on the trophy\'s rarity):**\n\n> Virat Kohli’s individual awards are so numerous, they’ve formed a support group for \'Trophies Who\'ve Never Met a Team Trophy.\' Their meetings are mostly just sharing stories about